In [1]:
import pandas as pd
import sys, os, time
os.environ['PYSPARK_PYTHON'] = os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
timestamp= str(time.time())

In [2]:
# notebook parameters, will be replaced by script arguments
application_name = "Technology Setup Test"
s3a_access_key = os.environ.get("s3a_access_key")
s3a_secret_key = os.environ.get("s3a_secret_key")
input_file_1 = "s3a://uga-data-lake/customer.csv"

In [3]:
import pyspark
from pyspark import SparkContext, SparkConf, SQLContext
from pyspark.sql import SparkSession
import pyspark.sql.types as T
import pyspark.sql.functions as F

In [4]:
spark = (SparkSession.builder.appName(application_name)
         .master("local[1]")
         .config('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.5.0')
         .config('spark.hadoop.fs.s3a.access.key', s3a_access_key)
         .config("spark.hadoop.fs.s3a.secret.key", s3a_secret_key)
         .config("spark.hadoop.fs.s3a.endpoint", "s3.us-east-2.amazonaws.com")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")
spark

:: loading settings :: url = jar:file:/usr/local/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-da68af00-e54a-49d7-8770-d814213d98a5;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.5.0 in central
	found software.amazon.awssdk#bundle;2.35.4 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.3.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.2.5.Final in central
:: resolution report :: resolve 169ms :: artifacts dl 7ms
	:: modules in use:
	org.apache.hadoop#hadoop-aws;3.5.0 from central in [default]
	org.wildfly.openssl#wildfly-openssl;2.2.5.Final from central in [default]
	software.amazon.awssdk#bundle;2.35.4 from central in [default]
	software.amazon.s3.analyt

In [5]:
customers = spark.read.option("inferSchema", "true").option("header", "true").csv(input_file_1)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


In [6]:
customers.show(3) # first three customers

+-----------+--------+----------+---------+--------------------+----------+------+------------------+-----------------+
|customer_id|store_id|first_name|last_name|               email|address_id|active|       create_date|      last_update|
+-----------+--------+----------+---------+--------------------+----------+------+------------------+-----------------+
|        1.0|     1.0|      MARY|    SMITH|MARY.SMITH@sakila...|       5.0|   1.0|2/14/2006 22:04:36|2/15/2006 4:57:20|
|        2.0|     1.0|  PATRICIA|  JOHNSON|PATRICIA.JOHNSON@...|       6.0|   1.0|2/14/2006 22:04:36|2/15/2006 4:57:20|
|        3.0|     1.0|     LINDA| WILLIAMS|LINDA.WILLIAMS@sa...|       7.0|   1.0|2/14/2006 22:04:36|2/15/2006 4:57:20|
+-----------+--------+----------+---------+--------------------+----------+------+------------------+-----------------+
only showing top 3 rows


In [7]:
customers.write.mode("overwrite").csv(f"tmp/customers_{timestamp}.csv")

In [8]:
customersRdd = sc.textFile(f"tmp/customers_{timestamp}.csv")

In [9]:
customersRdd.take(3)

['1.0,1.0,MARY,SMITH,MARY.SMITH@sakilacustomer.org,5.0,1.0,2/14/2006 22:04:36,2/15/2006 4:57:20',
 '2.0,1.0,PATRICIA,JOHNSON,PATRICIA.JOHNSON@sakilacustomer.org,6.0,1.0,2/14/2006 22:04:36,2/15/2006 4:57:20',
 '3.0,1.0,LINDA,WILLIAMS,LINDA.WILLIAMS@sakilacustomer.org,7.0,1.0,2/14/2006 22:04:36,2/15/2006 4:57:20']

In [10]:
customersRdd.saveAsTextFile(f"tmp/customers_{timestamp}.txt")